In [5]:
# 1. Clone repository & install ONLY missing helper packages
!git clone https://github.com/ahsan-c0ding/S4-Enhancement-Exploration.git
%cd S4-Enhancement-Exploration
!git checkout python

import os
import sys
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

# FIXED: Only install lightweight missing packages (Do NOT reinstall torch/torchvision)
%pip install einops coloredlogs torchinfo --quiet
%pip install git+https://github.com/mwalmsley/galaxy_mnist.git@c1fe9853a00bc34b2ff082585c6bb1654d34d239 --quiet

# Symlink workaround for model_params path resolving
parent_dir = os.path.dirname(current_dir)
_shim_path = os.path.join(parent_dir, "model_params")
_real_path = os.path.join(current_dir, "model_params")
if not os.path.exists(_shim_path) and os.path.exists(_real_path):
    os.symlink(_real_path, _shim_path)

Cloning into 'S4-Enhancement-Exploration'...
remote: Enumerating objects: 1152, done.
remote: Counting objects: 100% (220/220), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 1152 (delta 88), reused 180 (delta 68), pack-reused 932 (from 2)
Receiving objects: 100% (1152/1152), 94.54 MiB | 40.61 MiB/s, done.
Resolving deltas: 100% (489/489), done.
/kaggle/working/S4-Enhancement-Exploration
Branch 'python' set up to track remote branch 'python' from 'origin'.
Switched to a new branch 'python'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 3.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
  Preparing metadata (setup.py) ... done
Note: you may need to restart the kernel to use updated packages.


In [6]:
import math
import time
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from einops import repeat

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from model.functions import load_data

warnings.filterwarnings("ignore")
sns.set_style("darkgrid")
plt.rcParams["figure.figsize"] = [11, 6]

# Reproducibility
RNG_SEED = 30485
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(RNG_SEED)

CLASS_NAMES = ["Smooth Round", "Smooth Cigar", "Edge-on Disk", "Unbarred Spiral"]

# ============================================================================
# PRODUCTION S4D CONFIGURATION
# ============================================================================
COLORED = True            # RGB input (3-channel)
IN_CHANNELS = 3 if COLORED else 1

S4D_STATE = 64             # Hidden dimension / d_model
S4D_NUM_LAYERS = 3         # Stacked S4D layers
S4D_PATCH_SIZE = 4         # Patch resolution (4x4 patches)
S4D_POOLING = "last"       # Sequence pooling strategy
S4D_USE_NORM = False
S4D_USE_RESIDUAL = False
S4D_DROPOUT = 0.2
S4D_PATCH_EMBED = "conv"   # ConvPatchStem embedding

S4D_BATCH_SIZE = 32
S4D_FINAL_EPOCHS = 630     # Production epoch budget
LEARNING_RATE = 1e-3

print(f"Device: {DEVICE}")
print(f"Configured S4D Model: {S4D_NUM_LAYERS} layers | d_model={S4D_STATE} | patch_size={S4D_PATCH_SIZE} | stem={S4D_PATCH_EMBED}")

/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Device: cuda
Configured S4D Model: 3 layers | d_model=64 | patch_size=4 | stem=conv


In [7]:
class HilbertScan(nn.Module):
    """Reorders patches of a (B, C, H, W) image along a Hilbert curve."""
    def __init__(self, image_size=64, patch_size=1):
        super().__init__()
        assert image_size % patch_size == 0, "image_size must be divisible by patch_size"
        self.image_size = image_size
        self.patch_size = patch_size
        self.grid_size = image_size // patch_size
        self.num_patches = self.grid_size ** 2
        self.register_buffer("indices", self._get_hilbert_indices(self.grid_size))

    @staticmethod
    def _rot(s, x, y, rx, ry):
        if ry == 0:
            if rx == 1:
                x = s - 1 - x
                y = s - 1 - y
            x, y = y, x
        return x, y

    def _d2xy(self, n, d):
        x = y = 0
        t, s = d, 1
        while s < n:
            rx = (t // 2) & 1
            ry = (t ^ rx) & 1
            x, y = self._rot(s, x, y, rx, ry)
            x += s * rx
            y += s * ry
            t //= 4
            s *= 2
        return x, y

    def _get_hilbert_indices(self, grid_size):
        indices = []
        for d in range(grid_size * grid_size):
            x, y = self._d2xy(grid_size, d)
            indices.append(y * grid_size + x)
        return torch.LongTensor(indices)

    def forward(self, x):
        B, C, H, W = x.shape
        p = self.patch_size
        patches = x.unfold(2, p, p).unfold(3, p, p)
        patches = patches.permute(0, 2, 3, 1, 4, 5).contiguous()
        patches = patches.view(B, self.num_patches, C * p * p)
        return patches[:, self.indices, :]


class TakeLastTimestep(nn.Module):
    def forward(self, x):
        return x[:, -1, :]


class S4DConv(nn.Module):
    """Fast FFT-based parallel convolution S4D layer."""
    def __init__(self, d_model, d_state=64, dt_min=0.001, dt_max=0.1, transposed=True, lr=None):
        super().__init__()
        self.h = d_model
        self.n = d_state
        self.transposed = transposed

        log_dt = torch.rand(self.h) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
        log_A_real = torch.log(0.5 * torch.ones(self.h, self.n // 2))
        A_imag = math.pi * repeat(torch.arange(self.n // 2), 'n -> h n', h=self.h)
        C_init = torch.randn(self.h, self.n // 2, dtype=torch.cfloat)

        self.register("log_dt", log_dt, lr)
        self.register("log_A_real", log_A_real, lr)
        self.register("A_imag", A_imag, lr)

        self.C = nn.Parameter(torch.view_as_real(C_init))
        self.D = nn.Parameter(torch.randn(self.h))

    def register(self, name, tensor, lr=None):
        if lr == 0.0:
            self.register_buffer(name, tensor)
        else:
            self.register_parameter(name, nn.Parameter(tensor))
            optim = {"weight_decay": 0.0}
            if lr is not None:
                optim["lr"] = lr
            setattr(getattr(self, name), "_optim", optim)

    def forward(self, u):
        if not self.transposed:
            u = u.transpose(-1, -2)
        L = u.size(-1)

        dt = torch.exp(self.log_dt)
        C = torch.view_as_complex(self.C)
        A = -torch.exp(self.log_A_real) + 1j * self.A_imag

        dtA = A * dt.unsqueeze(-1)
        K_exp = torch.exp(dtA.unsqueeze(-1) * torch.arange(L, device=u.device))
        C_tilde = C * (torch.exp(dtA) - 1.) / A
        k = 2 * torch.einsum('hn, hnl -> hl', C_tilde, K_exp).real

        k_f = torch.fft.rfft(k, n=2 * L)
        u_f = torch.fft.rfft(u, n=2 * L)
        y = torch.fft.irfft(u_f * k_f, n=2 * L)[..., :L]
        y = y + u * self.D.unsqueeze(-1)

        if not self.transposed:
            y = y.transpose(-1, -2)
        return y, None


class ConvPatchStem(nn.Module):
    """Convolutional stem for local neighborhood mixing before patch projection."""
    def __init__(self, in_channels, d_model, patch_size):
        super().__init__()
        mid_channels = max(in_channels * 8, 32)
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, stride=1, padding=1),
            nn.GELU(),
            nn.Conv2d(mid_channels, d_model, kernel_size=patch_size, stride=patch_size),
        )

    def forward(self, x):
        return self.net(x)


class GalaxyClassifierS4DFast(nn.Module):
    """Production S4D Classifier."""
    def __init__(self, s4_state=64, d_model=64, num_classes=4, colored=True,
                 num_layers=2, patch_size=1, pooling="last",
                 use_norm=False, use_residual=False, dropout=0.0,
                 patch_embed="linear"):
        super().__init__()
        self.hilbert_channels = 1 if not colored else 3
        self.patch_size = patch_size
        self.pooling = pooling
        self.use_norm = use_norm
        self.use_residual = use_residual
        self.patch_embed = patch_embed

        if patch_embed == "linear":
            self.hilbert_scan = HilbertScan(image_size=64, patch_size=patch_size)
            patch_dim = self.hilbert_channels * patch_size * patch_size
            self.uproject = nn.Linear(patch_dim, d_model)
            self.conv_stem = None
        elif patch_embed == "conv":
            self.conv_stem = ConvPatchStem(self.hilbert_channels, d_model, patch_size)
            self.hilbert_scan = HilbertScan(image_size=64 // patch_size, patch_size=1)
            self.uproject = nn.Identity()

        self.s4_layers = nn.ModuleList([
            S4DConv(d_model=d_model, d_state=s4_state, transposed=False)
            for _ in range(num_layers)
        ])
        self.acts = nn.ModuleList([nn.GELU() for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(num_layers)]) if use_norm else None
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        if pooling == "last":
            self.take_last = TakeLastTimestep()
        elif pooling == "mean":
            self.take_last = None
        else:
            raise ValueError(f"Unknown pooling type {pooling}")

        self.fc = nn.Linear(d_model, num_classes)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, return_logits=True):
        if self.patch_embed == "conv":
            feat = self.conv_stem(x)
            x_seq = self.hilbert_scan(feat)
            h = self.uproject(x_seq)
        else:
            x_seq = self.hilbert_scan(x)
            h = self.uproject(x_seq)

        for i, (s4_layer, act) in enumerate(zip(self.s4_layers, self.acts)):
            residual = h
            h_in = self.norms[i](h) if self.use_norm else h
            h_out, _ = s4_layer(h_in)
            h_out = act(h_out)
            h_out = self.drop(h_out)
            h = residual + h_out if self.use_residual else h_out

        pooled = h.mean(dim=1) if self.take_last is None else self.take_last(h)
        logits = self.fc(pooled)

        if return_logits:
            return logits
        return self.softmax(logits)

In [8]:
# Load dataset
X, y_onehot, y = load_data(root="./data", download=True, train=True, colored=COLORED)
X_test, y_test_onehot, y_test = load_data(root="./data", download=True, train=False, colored=COLORED)
NUM_CLASSES = y_onehot.shape[1]

# FIXED: Unpack 4 outputs corresponding to passing X and y
x_train, x_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RNG_SEED, stratify=y
)

# Custom dataset supporting rotation and flip augmentation
class GalaxyDataset(Dataset):
    def __init__(self, images, labels, augment=False):
        self.images = images
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        lbl = self.labels[idx]
        if self.augment:
            # Random orthogonal rotation
            k = random.randint(0, 3)
            img = torch.rot90(img, k, [1, 2])
            # Random flips
            if random.random() > 0.5:
                img = torch.flip(img, [2])
            if random.random() > 0.5:
                img = torch.flip(img, [1])
        return img, lbl

train_dataset = GalaxyDataset(x_train, y_train, augment=True)
val_dataset = GalaxyDataset(x_val, y_val, augment=False)
test_dataset = GalaxyDataset(X_test, y_test, augment=False)

train_loader = DataLoader(train_dataset, batch_size=S4D_BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=S4D_BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=S4D_BATCH_SIZE, shuffle=False)

print(f"Data Split -> Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

100%|██████████| 68.7M/68.7M [00:06<00:00, 11.3MB/s]
100%|██████████| 17.3M/17.3M [00:01<00:00, 10.2MB/s]


Original Dataset Size: 8000 samples
Original Dataset Size: 2000 samples
Data Split -> Train: 6400 | Val: 1600 | Test: 2000


In [9]:
def create_s4d_optimizer(model, lr=1e-3, weight_decay=0.01):
    decay_params, no_decay_params, special_params = [], [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if hasattr(param, "_optim"):
            opt_dict = param._optim
            param_lr = opt_dict.get("lr", lr)
            special_params.append({'params': [param], 'lr': param_lr, 'weight_decay': 0.0})
        elif any(k in name for k in ["bias", "norm", "LayerNorm"]):
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    groups = [
        {'params': decay_params, 'weight_decay': weight_decay, 'lr': lr},
        {'params': no_decay_params, 'weight_decay': 0.0, 'lr': lr},
    ] + special_params
    return torch.optim.AdamW(groups)


def train_s4d_production(model, train_loader, val_loader, epochs=S4D_FINAL_EPOCHS, lr=LEARNING_RATE, device=DEVICE):
    model.to(device)
    optimizer = create_s4d_optimizer(model, lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-5
    )
    criterion = nn.CrossEntropyLoss()

    history = {"train_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0
    os.makedirs("checkpoints", exist_ok=True)
    best_ckpt_path = "checkpoints/s4d_production_best_d64.pt"

    start_time = time.time()
    print(f"Starting S4D production training for {epochs} epochs...")

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(images, return_logits=True)
            loss = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(-1) == labels).sum().item()
            total += labels.size(0)

        scheduler.step()
        train_loss = running_loss / total
        train_acc = correct / total

        # Validation
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                logits = model(images, return_logits=True)
                val_correct += (logits.argmax(-1) == labels).sum().item()
                val_total += labels.size(0)

        val_acc = val_correct / val_total
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_ckpt_path)

        if (epoch + 1) % 25 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch+1:03d}/{epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} (Best: {best_val_acc:.4f})")

    elapsed = time.time() - start_time
    print(f"\nTraining completed in {elapsed/60:.2f} mins. Best Validation Accuracy: {best_val_acc*100:.2f}%")
    
    # Load best checkpoint
    model.load_state_dict(torch.load(best_ckpt_path))
    return model, history

In [ ]:
# Instantiate model
model = GalaxyClassifierS4DFast(
    s4_state=S4D_STATE,
    d_model=S4D_STATE,
    num_classes=NUM_CLASSES,
    colored=COLORED,
    num_layers=S4D_NUM_LAYERS,
    patch_size=S4D_PATCH_SIZE,
    pooling=S4D_POOLING,
    use_norm=S4D_USE_NORM,
    use_residual=S4D_USE_RESIDUAL,
    dropout=S4D_DROPOUT,
    patch_embed=S4D_PATCH_EMBED
)

# Train production model
best_model, history = train_s4d_production(model, train_loader, val_loader, epochs=S4D_FINAL_EPOCHS)

Starting S4D production training for 630 epochs...
Epoch 025/630 | Train Loss: 0.6567 | Train Acc: 0.7220 | Val Acc: 0.7531 (Best: 0.7606)
Epoch 050/630 | Train Loss: 0.5778 | Train Acc: 0.7548 | Val Acc: 0.7731 (Best: 0.7750)
Epoch 075/630 | Train Loss: 0.5619 | Train Acc: 0.7602 | Val Acc: 0.7781 (Best: 0.7837)


In [ ]:
# Final Test Set Evaluation
best_model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        logits = best_model(images, return_logits=True)
        preds = logits.argmax(-1).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(labels.numpy())

test_acc = accuracy_score(all_targets, all_preds)
test_f1 = f1_score(all_targets, all_preds, average="macro")

print("=" * 50)
print(f"Final Production Test Accuracy: {test_acc * 100:.2f}%")
print(f"Final Production Test Macro-F1: {test_f1:.4f}")
print("=" * 50)

# Plot Training Curves & Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss & Accuracy Curves
axes[0].plot(history["train_acc"], label="Train Acc")
axes[0].plot(history["val_acc"], label="Val Acc")
axes[0].set_title("S4D Production Training Curves")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

# Confusion Matrix
cm = confusion_matrix(all_targets, all_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title("Test Set Confusion Matrix")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

plt.tight_layout()
plt.show()

In [11]:
import os
import shutil
from IPython.display import FileLink, display

# 1. Define the source and Kaggle's root working directory
source_path = "checkpoints/s4d_production_best_d64.pt"
destination_path = "/kaggle/working/s4d_production_best_d64.pt"

# 2. Copy the file to the root directory to avoid the Kaggle UI folder bug
if os.path.exists(source_path):
    shutil.copy(source_path, destination_path)
    print("✅ Model weights successfully moved to the root output folder!")
    print("Click the link below to download your weights:")
    
    # 3. Change directory temporarily to /kaggle/working to ensure FileLink works perfectly
    os.chdir('/kaggle/working')
    display(FileLink('s4d_production_best_d64.pt'))
    
    # Return to the repo directory just in case you run more cells afterward
    os.chdir('/kaggle/working/S4-Enhancement-Exploration')
else:
    print("❌ Error: Could not find the model checkpoint. Did the training complete successfully?")

✅ Model weights successfully moved to the root output folder!
Click the link below to download your weights:


/kaggle/working/s4d_production_best_d64.pt